In [0]:
# ============================================================
# Notebook 03: PySpark Basics
# Author: Alejandra Kheng
# Description: Demonstrates PySpark fundamentals including
#              reading data, filtering, groupBy aggregations,
#              and writing to Delta. Shows Python-based
#              data engineering patterns used in production.
# ============================================================

# Read the projects Delta table into a PySpark DataFrame
df_projects = spark.table("healthcare_analytics.projects")
df_departments = spark.table("healthcare_analytics.departments")

# Show the schema
print("Projects schema:")
df_projects.printSchema()

# Preview the data
print("Projects data:")
df_projects.show()

Projects schema:
root
 |-- project_id: integer (nullable = true)
 |-- project_name: string (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- completed_tasks: integer (nullable = true)
 |-- total_tasks: integer (nullable = true)
 |-- due_date: date (nullable = true)

Projects data:
+----------+--------------------+-------------+---------------+-----------+----------+
|project_id|        project_name|department_id|completed_tasks|total_tasks|  due_date|
+----------+--------------------+-------------+---------------+-----------+----------+
|       101|       EHR Migration|            1|             45|         50|2024-03-01|
|       102|Patient Intake Re...|            1|             20|         50|2024-04-15|
|       103|BI Dashboard Rollout|            2|             48|         50|2024-02-28|
|       104| Claims Data Cleanup|            2|             30|         50|2024-05-01|
|       105|       Audit Prep Q1|            3|             50|         50|2024-01-31|
|  

In [0]:
# ============================================================
# Filter, transform, and aggregate with PySpark
# ============================================================

from pyspark.sql.functions import col, round as spark_round, avg, count, when

# Add completion percentage column
df_with_pct = df_projects.withColumn(
    "completion_pct",
    spark_round(col("completed_tasks") / col("total_tasks") * 100, 1)
)

# Filter: only projects above 60% completion
df_on_track = df_with_pct.filter(col("completion_pct") >= 60)

print("Projects on track (60%+ completion):")
df_on_track.select(
    "project_id",
    "project_name",
    "completed_tasks",
    "total_tasks",
    "completion_pct"
).show()

# GroupBy: average completion rate per department
print("Average completion rate by department:")
df_with_pct.groupBy("department_id") \
    .agg(
        count("project_id").alias("total_projects"),
        spark_round(avg("completion_pct"), 1).alias("avg_completion_pct")
    ) \
    .orderBy("department_id") \
    .show()

Projects on track (60%+ completion):
+----------+--------------------+---------------+-----------+--------------+
|project_id|        project_name|completed_tasks|total_tasks|completion_pct|
+----------+--------------------+---------------+-----------+--------------+
|       101|       EHR Migration|             45|         50|          90.0|
|       103|BI Dashboard Rollout|             48|         50|          96.0|
|       104| Claims Data Cleanup|             30|         50|          60.0|
|       105|       Audit Prep Q1|             50|         50|         100.0|
|       106|   Compliance Review|             35|         50|          70.0|
|       107|Data Quality Fram...|             42|         50|          84.0|
+----------+--------------------+---------------+-----------+--------------+

Average completion rate by department:
+-------------+--------------+------------------+
|department_id|total_projects|avg_completion_pct|
+-------------+--------------+------------------+
|  

In [0]:
# ============================================================
# Write transformed data back to Delta
# ============================================================

# Join projects with departments for a final summary DataFrame
df_summary = df_with_pct.join(
    df_departments,
    on="department_id",
    how="left"
).select(
    "project_id",
    "project_name",
    "department_name",
    "completed_tasks",
    "total_tasks",
    "completion_pct"
).withColumn(
    "performance_tier",
    when(col("completion_pct") >= 90, "High")
    .when(col("completion_pct") >= 60, "On Track")
    .otherwise("At Risk")
)

# Write to Delta table
df_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_analytics.project_summary")

# Verify final output
print("Final project summary Delta table:")
spark.table("healthcare_analytics.project_summary") \
    .orderBy("department_name", "completion_pct") \
    .show()

Final project summary Delta table:
+----------+--------------------+-------------------+---------------+-----------+--------------+----------------+
|project_id|        project_name|    department_name|completed_tasks|total_tasks|completion_pct|performance_tier|
+----------+--------------------+-------------------+---------------+-----------+--------------+----------------+
|       102|Patient Intake Re...|Clinical Operations|             20|         50|          40.0|         At Risk|
|       101|       EHR Migration|Clinical Operations|             45|         50|          90.0|            High|
|       108|   Report Automation|   Data & Analytics|             15|         50|          30.0|         At Risk|
|       107|Data Quality Fram...|   Data & Analytics|             42|         50|          84.0|        On Track|
|       104| Claims Data Cleanup| Health Informatics|             30|         50|          60.0|        On Track|
|       103|BI Dashboard Rollout| Health Informatics|